In [ ]:
import pandas as pd
import numpy as np
o = pd.read_csv("orders.csv", parse_dates=["order_date"])
p = pd.read_csv("payments.csv", parse_dates=["created_at"])
r = pd.read_csv("refunds.csv", parse_dates=["refund_date"])
s = pd.read_csv("settlements.csv", parse_dates=["payout_date"])
pr = pd.read_csv("products.csv")
for n, d in {"orders": o, "payments": p, "refunds": r, "settlements": s, "products": pr}.items():
    print(n, d.shape, "duplicate rows:", d.duplicated().sum(), "nulls:", d.isna().sum().sum())
print(o.status.value_counts().to_dict(), p.status.value_counts().to_dict(), s.type.value_counts().to_dict())
print("unique orders:", o.order_id.nunique(), "| payments without order:", (~p.order_id.isin(o.order_id)).sum())

In [ ]:
raw = o.order_total.sum()
o = o.drop_duplicates()
o = o.merge(pr[["product_id", "unit_cost"]], on="product_id", how="left", validate="m:1")
o["cogs"] = o.quantity * o.unit_cost
print(o.order_id.is_unique, round(raw - o.order_total.sum(), 2), o.cogs.isna().sum())

In [ ]:
cap = p[p.status == "captured"]
paid = cap.groupby("order_id").agg(captured=("amount", "sum"), n_cap=("amount", "size"), last_cap=("created_at", "max"), payment_ids=("payment_id", ",".join))
print(round(p.amount.sum() - cap.amount.sum(), 2), (paid.n_cap > 1).sum())

In [ ]:
pm = p.set_index("payment_id").order_id
r["order_id"] = r.payment_id.map(pm)
s["order_id"] = s.payment_id.map(pm)
refd = r.groupby("order_id").amount.sum().rename("refunded")
stl = s.groupby("order_id").agg(fee=("fee", "sum"), settled=("net", "sum"))
charge = s[s.type == "charge"]
lag = (charge.payout_date - charge.payment_id.map(p.set_index("payment_id").created_at).dt.normalize()).dt.days
L = o.set_index("order_id")[["status", "order_date", "order_total", "cogs"]].join([paid, refd, stl], how="outer")
L = L.fillna({"order_total": 0, "captured": 0, "n_cap": 0, "refunded": 0, "fee": 0})
L["net_revenue"] = np.where(L.status == "cancelled", 0, L.order_total - L.refunded)
L["cash_diff"] = (L.captured - L.refunded - L.net_revenue).round(2)
L["settle_diff"] = (L.captured - L.refunded - L.fee - L.settled.fillna(0)).round(2)
print("unmatched refunds:", r.order_id.isna().sum(), "| unmatched settlements:", s.order_id.isna().sum())
print("payout lag (days):", lag.value_counts().to_dict(), "| last payout:", s.payout_date.max().date())
print("ledger rows:", len(L), L.index.is_unique, "| lost cash:", round(L.captured.sum() - cap.amount.sum(), 2), "| lost sales:", round(L.order_total.sum() - o.order_total.sum(), 2))
print("cash_diff rows:", (L.cash_diff != 0).sum(), "| settle_diff rows:", (L.settle_diff != 0).sum())

In [ ]:
k = {}
k["unknown_order"] = L.status.isna()
k["double_charge"] = L.n_cap > 1
k["unpaid"] = L.captured.eq(0) & L.cash_diff.ne(0)
k["cancel_no_refund"] = L.status.eq("cancelled") & L.cash_diff.ne(0)
k["amount_mismatch"] = L.cash_diff.ne(0)
k["in_transit"] = L.settled.isna() & L.last_cap.ge("2025-12-30")
k["no_settlement"] = L.settled.isna() & L.captured.gt(0)
k["short_settlement"] = L.settle_diff.ne(0)
L["recon"] = np.select(list(k.values()), list(k), "reconciled")
L.groupby("recon")[["cash_diff", "settle_diff"]].agg(["count", "sum"])

In [ ]:
done = L.status.eq("completed")
fp = pd.Series({"gross_sales": L.order_total[L.status.ne("cancelled")].sum(),
                "refunds_on_sales": L.refunded[L.status.ne("cancelled")].sum(),
                "net_revenue": L.net_revenue.sum(),
                "revenue_without_cost": L.net_revenue[done & L.cogs.isna()].sum(),
                "cogs": L.cogs[done].sum(),
                "cash_captured": L.captured.sum(),
                "all_refunds": L.refunded.sum(),
                "processor_fees": L.fee.sum(),
                "settled_cash": L.settled.sum(),
                "unsettled_difference": L.settle_diff.sum()})
fp["gross_margin"] = fp.net_revenue - fp.revenue_without_cost - fp.cogs
fp.round(2)

In [ ]:
print(round(cap.amount.sum() - r.amount.sum() - fp.net_revenue, 2), round(L.cash_diff.sum(), 2))
print(round(cap.amount.sum() - r.amount.sum() - s.fee.sum() - s.net.sum(), 2), round(L.settle_diff.sum(), 2))

In [ ]:
act = {"in_transit": "timing", "double_charge": "refund_owed", "cancel_no_refund": "refund_owed", "unpaid": "collect"}
L["resolution"] = L.recon.map(act).fillna("unexplained").where(L.recon != "reconciled", "none")
L.reset_index(names="order_id").to_csv("transaction_ledger.csv", index=False)
L[L.recon != "reconciled"].reset_index(names="order_id").to_csv("unresolved_items.csv", index=False)
fp.round(2).rename("amount").to_csv("financial_position.csv")
o.to_csv("orders.csv", index=False)
r.to_csv("refunds.csv", index=False)
s.to_csv("settlements.csv", index=False)
L.groupby("resolution")[["cash_diff", "settle_diff"]].sum()

## Summary
- Net revenue $95,436.50; margin $40,814.24 ($6,746.75 without cost).
- Cash gap $1,129.87: $1,402 owed back, $693 unpaid, $420.87 unexplained.
- Settlement gap $2,921: $2,225 in transit, $696 unexplained.
- Input files generated with AI.